# Taller 7: Estructuras de datos básicas — listas, matrices, diccionarios y una primera mirada a conjuntos (continuación)

Continuación del Taller 3. Ahí trabajaste listas, matrices y diccionarios con las operaciones más básicas. Aquí subimos un peldaño, sobre todo en **listas** (que siguen siendo la estructura donde más vale afianzar la mecánica), profundizamos un poco más en matrices y diccionarios, y damos una **primera mirada** a los conjuntos (`set`) — sin entrar todavía en todas sus operaciones, solo lo suficiente para resolver un problema muy común.

> Cada ejercicio trae **enunciado**, **modelo de pensamiento** y **solución con type hints**, seguida de un análisis de **complejidad temporal y espacial**.

## Ejercicio 1: Mecánica de listas: slicing con paso y modificación in-place

### Enunciado

Tienes `temperaturas`, una lista con 24 valores (uno por cada hora del día). Resuelve, en este orden: (a) obtén las temperaturas de las horas **pares** usando slicing con paso, sin ciclo; (b) reemplaza las primeras 3 horas (posiciones 0, 1 y 2) por tres valores nuevos, usando asignación por slice (sin recrear la lista completa); (c) elimina la última temperatura con `.pop()` y guarda ese valor eliminado en una variable; (d) inserta un valor nuevo en la posición 5 con `.insert()`, sin sobreescribir lo que ya había ahí.

### Modelo de pensamiento

1. El slicing con paso `lista[inicio:fin:paso]` generaliza lo que ya conoces del Taller 3: paso 2 significa "toma un elemento de cada dos", empezando en `inicio`. Es útil cuando quieres "cada n-ésimo elemento" sin escribir un ciclo con `range(0, len(lista), 2)`.
2. Asignar a un slice (`lista[0:3] = [...]`) es distinto de solo leerlo: en vez de crear una lista nueva, **modifica in-place** el tramo indicado de la lista original. El tramo de la derecha ni siquiera necesita tener el mismo tamaño que el de la izquierda — si tuviera más o menos elementos, la lista completa crecería o se encogería.
3. `.pop()` sin argumento elimina y **devuelve** el último elemento — a diferencia de `.remove(valor)` (que busca un valor específico y lo elimina sin importar la posición) o de `del lista[-1]` (que elimina pero no te entrega el valor).
4. `.insert(posicion, valor)` desplaza todos los elementos desde esa posición hacia la derecha para hacerle espacio al nuevo valor — no sobreescribe nada, a diferencia de asignar directamente `lista[posicion] = valor`.

In [1]:
temperaturas: list[float] = [
    18, 17, 16, 16, 17, 19, 21, 23, 25, 26, 27, 28,
    28, 27, 26, 24, 22, 20, 19, 18, 18, 17, 17, 16,
]

# (a) horas pares, sin ciclo
temperaturas_horas_pares: list[float] = temperaturas[0::2]

# (b) reemplazar las primeras 3 horas in-place
temperaturas[0:3] = [15.0, 15.5, 16.0]

# (c) eliminar y guardar la última temperatura
ultima_temperatura: float = temperaturas.pop()

# (d) insertar un valor nuevo en la posición 5
temperaturas.insert(5, 30.0)

print(temperaturas_horas_pares)
print(temperaturas[:8])
print("Última eliminada:", ultima_temperatura)


[18, 16, 17, 21, 25, 27, 28, 26, 22, 19, 18, 17]
[15.0, 15.5, 16.0, 16, 17, 30.0, 19, 21]
Última eliminada: 16


**Complejidad:** el slicing con paso `lista[0::2]` recorre y copia aproximadamente la mitad de la lista → **O(n) en tiempo y en espacio**, con `n = len(lista)`. La asignación por slice sobre un tramo de tamaño fijo (3 posiciones) es O(1) en este caso, aunque en general depende del tamaño del tramo reemplazado. `.pop()` sin argumento es **O(1)** (elimina del final, no hay que desplazar nada). `.insert(posicion, valor)` es **O(n)** en el peor caso, porque debe desplazar una posición hacia la derecha a todos los elementos que están después de `posicion`.

## Ejercicio 2: Transformar una lista en otra, con una condición

### Enunciado

Dada una lista de precios, construye una **nueva** lista aplicando un descuento del 10% únicamente a los precios mayores a $100.000; los demás precios quedan sin cambios.

### Modelo de pensamiento

1. Este ejercicio combina dos patrones que ya conoces por separado: "construir una lista nueva a partir de otra" (Taller 3) y "decidir con una condición" (Taller 1 y 2). La condición vive **dentro** del ciclo: por cada precio, primero preguntas si supera el umbral, y según la respuesta decides qué valor agregar a la lista nueva.
2. Sigue sin modificarse la lista original — el patrón de siempre es: lista nueva vacía, recorrer la original, y en cada vuelta hacer `.append(...)` del valor que corresponda (con descuento o sin él).
3. A diferencia del Taller 3 (donde todos los precios recibían el mismo tratamiento), aquí **no todos los elementos pasan por la misma transformación** — parte del ejercicio es notar que el `if` decide *cuál* fórmula aplicar en cada vuelta, no si se agrega o no el elemento (los dos casos siempre agregan algo a la lista nueva, a diferencia de un filtro).

In [2]:
def aplicar_descuento_por_volumen(precios: list[float]) -> list[float]:
    precios_finales: list[float] = []
    for precio in precios:
        if precio > 100000:
            precios_finales.append(precio * 0.9)
        else:
            precios_finales.append(precio)
    return precios_finales


print(aplicar_descuento_por_volumen([50000.0, 150000.0, 90000.0, 200000.0]))
# [50000.0, 135000.0, 90000.0, 180000.0]


[50000.0, 135000.0, 90000.0, 180000.0]


**Complejidad:** un solo recorrido de la lista, con una comparación y una operación aritmética por elemento → **O(n) en tiempo**, con `n = len(precios)`. En **espacio**, se construye una lista nueva del mismo tamaño que la original → **O(n)**.

## Ejercicio 3: Insertar y eliminar manteniendo una condición

### Enunciado

Resuelve dos funciones relacionadas con listas: (a) `eliminar_menores_a(lista, umbral)`, que devuelva una **nueva** lista sin los elementos menores al umbral (cuidado: no elimines elementos de la lista mientras la recorres); (b) `insertar_ordenado(lista_ordenada, valor)`, que inserte `valor` en la posición correcta de una lista ya ordenada de menor a mayor, usando `.insert()`, sin volver a ordenar toda la lista desde cero.

### Modelo de pensamiento

1. Un error muy común es recorrer una lista con `for elemento in lista` e ir eliminando elementos de esa misma lista dentro del ciclo (`lista.remove(elemento)`): esto puede saltarse elementos, porque Python reacomoda los índices internos mientras la lista cambia de tamaño durante el recorrido. La forma segura es el patrón que ya conoces: construir una lista **nueva** con los elementos que sí quieres conservar, en vez de modificar la original mientras la recorres.
2. Para insertar en la posición correcta sin reordenar todo, primero hay que **encontrar esa posición**: recorre la lista buscando el primer índice donde el valor ya guardado es mayor o igual al que quieres insertar — ese es el lugar donde debe ir tu nuevo valor para que la lista siga ordenada.
3. Hay que contemplar el caso en que el ciclo termina sin encontrar ningún elemento mayor (el valor nuevo es el más grande de todos): en ese caso, la posición correcta es el final de la lista, así que conviene inicializar `posicion` en `len(lista_ordenada)` antes de empezar a buscar.

In [3]:
def eliminar_menores_a(lista: list[float], umbral: float) -> list[float]:
    resultado: list[float] = []
    for elemento in lista:
        if elemento >= umbral:
            resultado.append(elemento)
    return resultado


def insertar_ordenado(lista_ordenada: list[float], valor: float) -> list[float]:
    posicion = len(lista_ordenada)
    for indice in range(len(lista_ordenada)):
        if lista_ordenada[indice] >= valor:
            posicion = indice
            break
    lista_ordenada.insert(posicion, valor)
    return lista_ordenada


print(eliminar_menores_a([3.0, 7.0, 1.0, 9.0, 2.0], 3.0))  # [3.0, 7.0, 9.0]
print(insertar_ordenado([1, 3, 5, 8], 4))                   # [1, 3, 4, 5, 8]


[3.0, 7.0, 9.0]
[1, 3, 4, 5, 8]


**Complejidad:** `eliminar_menores_a` recorre la lista una vez → **O(n) en tiempo**, con `n = len(lista)`; en **espacio**, la lista resultado ocupa **O(k)**, donde `k` es la cantidad de elementos que quedan (en el peor caso, `k = n`). `insertar_ordenado` recorre la lista buscando la posición (hasta O(n) en el peor caso) y luego `.insert()` desplaza los elementos posteriores (también O(n) en el peor caso) → **O(n) en tiempo** en total; en **espacio** es **O(1)** adicional, porque la inserción ocurre in-place sobre la lista recibida.

## Ejercicio 4: Matriz: transponer

### Enunciado

Escribe una función `transponer(matriz)` que reciba una matriz rectangular (lista de listas, todas del mismo largo) y devuelva su transpuesta: la fila `i` de la matriz original se convierte en la columna `i` del resultado.

### Modelo de pensamiento

1. Si la matriz original tiene `f` filas y `c` columnas, la transpuesta tiene `c` filas y `f` columnas. Eso te dice de entrada que el ciclo externo del resultado debe recorrer **columnas** de la original (de 0 a `c - 1`), y el ciclo interno debe recorrer **filas** (de 0 a `f - 1`) para ir tomando el elemento correspondiente de cada una.
2. Para la nueva fila `j` del resultado, necesitas juntar el elemento en la posición `j` de **cada** fila original — es decir, `matriz[i][j]` para cada `i`. Ese doble recorrido (columna fija, todas las filas) es exactamente el ciclo anidado invertido respecto a cómo está guardada la matriz.
3. Antes de programar, calcula a mano un ejemplo pequeño (2×3) y anota a qué posición del resultado va cada `matriz[i][j]` — verificar la fórmula de índices con un ejemplo chico evita errores de "uno de más" (*off-by-one*) al programar el ciclo anidado.

In [4]:
def transponer(matriz: list[list[float]]) -> list[list[float]]:
    filas = len(matriz)
    columnas = len(matriz[0])

    resultado: list[list[float]] = []
    for j in range(columnas):
        nueva_fila: list[float] = []
        for i in range(filas):
            nueva_fila.append(matriz[i][j])
        resultado.append(nueva_fila)
    return resultado


matriz = [
    [1, 2, 3],
    [4, 5, 6],
]
for fila in transponer(matriz):
    print(fila)
# [1, 4]
# [2, 5]
# [3, 6]


[1, 4]
[2, 5]
[3, 6]


**Complejidad:** se visita cada una de las `f × c` posiciones de la matriz exactamente una vez → **O(f × c) en tiempo**, que también se puede escribir como **O(n)** si `n` es el número total de elementos de la matriz. En **espacio**, la matriz resultado tiene el mismo número de elementos que la original → **O(f × c)** adicional (no se modifica la matriz de entrada, se construye una nueva).

## Ejercicio 5: Diccionario invertido, con colisiones

### Enunciado

Dado un diccionario `nombre -> edad`, escribe una función `invertir(diccionario)` que devuelva un nuevo diccionario `edad -> lista_de_nombres`, agrupando **todos** los nombres que comparten la misma edad (no puedes simplemente invertir clave y valor uno a uno, porque varias personas pueden tener la misma edad, y un diccionario no permite claves repetidas).

### Modelo de pensamiento

1. La trampa del enunciado está en la palabra "invertir": si simplemente hicieras `invertido[edad] = nombre` para cada par, cada nueva persona con una edad ya vista **sobreescribiría** a la anterior, perdiendo información. Necesitas que cada clave del resultado apunte a una **lista** de nombres, no a un solo nombre.
2. Esto es el mismo problema de "inicializar antes de acumular" que ya conoces de los diccionarios de frecuencias: la primera vez que aparece una edad, tienes que crear la lista vacía correspondiente antes de poder hacerle `append`. `diccionario.setdefault(clave, [])` hace exactamente eso en una sola línea: si la clave no existe, la crea con el valor por defecto (una lista vacía) y la devuelve; si ya existe, simplemente devuelve la lista existente.
3. Recorres el diccionario original con `.items()` para obtener pares `(nombre, edad)` a la vez, en vez de recorrer solo las claves y volver a consultar el valor con `diccionario[nombre]` en cada vuelta.

In [5]:
def invertir(diccionario: dict[str, int]) -> dict[int, list[str]]:
    invertido: dict[int, list[str]] = {}
    for nombre, edad in diccionario.items():
        invertido.setdefault(edad, []).append(nombre)
    return invertido


edades = {"Ana": 20, "Luis": 22, "Marta": 20, "Pedro": 21, "Sofía": 22}
print(invertir(edades))
# {20: ['Ana', 'Marta'], 22: ['Luis', 'Sofía'], 21: ['Pedro']}


{20: ['Ana', 'Marta'], 22: ['Luis', 'Sofía'], 21: ['Pedro']}


**Complejidad:** un solo recorrido del diccionario de entrada, y cada `setdefault` + `append` es O(1) en promedio → **O(n) en tiempo**, con `n` el número de pares en el diccionario original. En **espacio**, el diccionario `invertido` termina guardando los mismos `n` nombres repartidos en listas, más una entrada por cada edad distinta `e` → **O(n)** en total.

## Ejercicio 6: Primer acercamiento a los conjuntos: eliminar duplicados preservando el orden

### Enunciado

Escribe una función `sin_duplicados(lista)` que devuelva una nueva lista con los mismos elementos, en el mismo orden de aparición, pero sin repetidos. (Usar directamente `list(set(lista))` no sirve porque los conjuntos no garantizan mantener el orden original — esa es justamente la diferencia que vas a explorar en este ejercicio.)

### Modelo de pensamiento

1. Un `set` (conjunto) en Python es una colección donde **no puede haber elementos repetidos**, y no garantiza ningún orden de iteración en particular. El requisito de "preservar el orden" descarta la solución obvia de `set(lista)` como resultado final — pero eso no significa que un `set` no sirva para nada aquí: puedes usarlo como **estructura auxiliar** dentro de tu propio ciclo con `for`.
2. La pregunta que necesitas responder muchas veces mientras recorres la lista es "¿ya vi este elemento antes?". Podrías preguntarlo con `elemento in lista_resultado` (revisando una lista), pero un `set` responde esa misma pregunta de forma mucho más directa: `elemento in un_set`.
3. El patrón final es: recorrer la lista original en orden; por cada elemento, si no está en el conjunto `vistos`, agregarlo tanto a la lista resultado como al conjunto `vistos` (con `.add(...)`); si ya está, lo ignoras y sigues. Es el mismo patrón de acumulador de siempre, con un conjunto adicional que lleva la cuenta de lo que ya procesaste.

In [6]:
def sin_duplicados(lista: list[int]) -> list[int]:
    vistos: set[int] = set()
    resultado: list[int] = []
    for elemento in lista:
        if elemento not in vistos:
            vistos.add(elemento)
            resultado.append(elemento)
    return resultado


print(sin_duplicados([3, 1, 3, 2, 1, 4, 2]))  # [3, 1, 2, 4]


[3, 1, 2, 4]


**Complejidad:** un solo recorrido de la lista, y cada operación dentro del ciclo (`in`, `add`, `append`) es O(1) en promedio → **O(n) en tiempo**, con `n = len(lista)`. En **espacio**, tanto `vistos` como `resultado` pueden llegar a guardar hasta `n` elementos en el peor caso (todos distintos) → **O(n)**.

## Ejercicio 7: Reporte de calificaciones por estudiante (reto)

### Enunciado

Tienes una lista de nombres de estudiantes y una matriz de notas: cada fila `i` de la matriz corresponde, en el mismo orden, al estudiante `i` de la lista de nombres; cada columna es un examen distinto. Escribe funciones que, combinadas, construyan un diccionario `nombre -> {"promedio": ..., "mejor_nota": ..., "peor_nota": ...}` para cada estudiante.

### Modelo de pensamiento

1. Identifica primero la sub-tarea que se repite: dado **un solo** estudiante (una fila de la matriz), calcular su promedio, su mejor y su peor nota. Aísla eso en una función auxiliar (`resumen_de_fila`) antes de pensar en la matriz completa — resolver el problema para un caso individual siempre es más fácil que resolverlo para todos a la vez.
2. Dentro de `resumen_de_fila` puedes calcular los tres valores en un solo recorrido de la fila, reutilizando patrones que ya conoces: "mejor/peor hasta ahora" (dos variables actualizadas en paralelo, una para el máximo y otra para el mínimo) y un acumulador de suma para el promedio — los tres se calculan sin necesitar tres ciclos separados.
3. Para conectar cada nombre con su fila correspondiente, recorre con índices (`for i in range(len(nombres))`): `nombres[i]` y `matriz_notas[i]` están en la misma posición porque así está construido el problema — es la misma idea de "dos listas en paralelo" que ya usaste antes.
4. La función principal (`resumen_por_estudiante`) no calcula nada directamente: solo recorre las posiciones, llama a `resumen_de_fila` con la fila que corresponde, y guarda ese resultado en el diccionario final bajo la llave del nombre correspondiente.

In [7]:
def resumen_de_fila(notas: list[float]) -> dict[str, float]:
    mejor = notas[0]
    peor = notas[0]
    suma = 0.0
    for nota in notas:
        if nota > mejor:
            mejor = nota
        if nota < peor:
            peor = nota
        suma += nota
    promedio = suma / len(notas)
    return {"promedio": promedio, "mejor_nota": mejor, "peor_nota": peor}


def resumen_por_estudiante(
    nombres: list[str], matriz_notas: list[list[float]]
) -> dict[str, dict[str, float]]:
    resumen: dict[str, dict[str, float]] = {}
    for i in range(len(nombres)):
        resumen[nombres[i]] = resumen_de_fila(matriz_notas[i])
    return resumen


nombres = ["Ana", "Luis", "Marta"]
notas = [
    [4.5, 3.9, 4.2],
    [2.5, 3.0, 2.8],
    [5.0, 4.8, 4.9],
]

print(resumen_por_estudiante(nombres, notas))


{'Ana': {'promedio': 4.2, 'mejor_nota': 4.5, 'peor_nota': 3.9}, 'Luis': {'promedio': 2.766666666666667, 'mejor_nota': 3.0, 'peor_nota': 2.5}, 'Marta': {'promedio': 4.9, 'mejor_nota': 5.0, 'peor_nota': 4.8}}


**Complejidad:** sea `e` el número de estudiantes y `m` el número de notas por estudiante. `resumen_de_fila` recorre una fila una vez → O(m). `resumen_por_estudiante` la llama una vez por cada estudiante → **O(e × m) en tiempo** en total, equivalente a visitar cada nota de la matriz exactamente una vez. En **espacio**, el diccionario resultado guarda una entrada pequeña (tres números) por cada uno de los `e` estudiantes → **O(e)**, muy por debajo del tamaño total de la matriz de notas.